# 02 · Capa Gold: grilla hexagonal H3 y análisis exploratorio espacial (AEDE)

Millones de píxeles no se analizan uno por uno con estadística espacial clásica.
Se agregan a una **grilla hexagonal discreta global H3** (Uber), nativa en Databricks SQL:

- Todos los hexágonos tienen la misma área y **6 vecinos equidistantes** (no hay ambigüedad torre/reina).
- La función `h3_longlatash3string` asigna cada píxel a su hexágono en paralelo, sin geometrías.
- Resolución 9 ≈ 0.105 km² (≈ 260 píxeles de 20 m por hexágono).

Al final se exporta el **dataset limpio** (Parquet, CSV y GeoJSON) que se usa en la demostración.

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
from pyspark.sql import functions as F
silver = spark.table(dbx.tbl("silver_s2_pixeles"))
hansen = spark.table(dbx.tbl("silver_hansen_pixeles"))
YEARS = sorted(r.year for r in silver.select("year").distinct().collect())
Y0, Y1 = YEARS[0], YEARS[-1]
R = C.H3_RES_FINE
silver.createOrReplaceTempView("px")
hansen.createOrReplaceTempView("hx")
print("Años disponibles:", YEARS)

## 1. Agregación píxel → hexágono (Spark SQL + H3 nativo)

Se define **píxel perturbado** como NDVI < 0.3 con señal de suelo desnudo (BSI > 0) o de agua
turbia (MNDWI > 0). En el corredor minero, esa combinación corresponde a playas de lavado y
pozas de relave: la huella típica de la minería aluvial.

In [ ]:
gold_anual = spark.sql(f"""
SELECT h3_longlatash3string(lon, lat, {R}) AS h3, year,
       COUNT(*)                         AS n_px,
       AVG(n_obs)                       AS n_obs,
       AVG(ndvi)  AS ndvi,  AVG(nbr) AS nbr,
       AVG(mndwi) AS mndwi, AVG(bsi) AS bsi,
       AVG(CASE WHEN ndvi < 0.3 AND (bsi > 0 OR mndwi > 0) THEN 1D ELSE 0D END) AS frac_pert,
       AVG(CASE WHEN mndwi > 0 THEN 1D ELSE 0D END)                              AS frac_agua
FROM px
GROUP BY 1, 2
""")
dbx.save_table(gold_anual, "gold_h3_anual")

# Universo de hexágonos y referencia Hansen (cobertura completa, sin nubes)
gold_hansen = spark.sql(f"""
SELECT h3_longlatash3string(lon, lat, {R}) AS h3,
       COUNT(*)            AS n_px_total,
       AVG(treecover2000)  AS treecover2000,
       AVG(CASE WHEN lossyear BETWEEN {Y0 - 2000 + 1} AND {Y1 - 2000} THEN 1D ELSE 0D END) AS frac_perdida_hansen,
       AVG(CASE WHEN lossyear BETWEEN 1 AND {Y1 - 2000} THEN 1D ELSE 0D END)             AS frac_perdida_2001
FROM hx
GROUP BY 1
HAVING n_px_total >= 150
""")
dbx.save_table(gold_hansen, "gold_h3_hansen")

# Referencia oficial peruana: Geobosques (MINAM). Solo pérdida antrópica; la natural (ríos) se reporta aparte
HAS_GB = spark.catalog.tableExists(dbx.tbl("silver_geobosques_pixeles"))
if HAS_GB:
    spark.table(dbx.tbl("silver_geobosques_pixeles")).createOrReplaceTempView("gbx")
    codes = ",".join(map(str, C.geobosques_codes(Y0, Y1)))
    gold_gb = spark.sql(f"""
    SELECT h3_longlatash3string(lon, lat, {R}) AS h3,
           AVG(CASE WHEN perdida_gb IN ({codes}) THEN 1D ELSE 0D END)  AS frac_perdida_oficial,
           AVG(CASE WHEN perdida_gb = 26 THEN 1D ELSE 0D END)          AS frac_perdida_natural_2025,
           AVG(CASE WHEN cobertura_gb = 4 THEN 1D ELSE 0D END)         AS frac_bosque_2025,
           AVG(CASE WHEN riesgo_gb BETWEEN 1 AND 5 THEN riesgo_gb END) AS riesgo_deforestacion
    FROM gbx GROUP BY 1""")
    dbx.save_table(gold_gb, "gold_h3_geobosques")
print("Geobosques disponible:", HAS_GB)

## 2. Tabla analítica ancha: un hexágono por fila

Un año se marca como faltante en un hexágono si menos del 50 % de sus píxeles tuvo al menos una
observación sin nube. Esos vacíos se rellenan después con **kriging** (notebook 04).

In [ ]:
ga = spark.table(dbx.tbl("gold_h3_anual")).alias("a")
gh = spark.table(dbx.tbl("gold_h3_hansen")).alias("h")
valid = (ga.join(gh, "h3")
         .withColumn("ok", F.col("n_px") / F.col("n_px_total") >= 0.5)
         .withColumn("ndvi", F.when(F.col("ok"), F.col("ndvi")))
         .withColumn("frac_pert", F.when(F.col("ok"), F.col("frac_pert"))))
wide = (valid.groupBy("h3").pivot("year", YEARS)
        .agg(F.first("ndvi").alias("ndvi"), F.first("frac_pert").alias("pert"), F.first("n_obs").alias("nobs")))
for y in YEARS:
    for m in ["ndvi", "pert", "nobs"]:
        wide = wide.withColumnRenamed(f"{y}_{m}", f"{m}_{y}")
base_h3 = gh.join(spark.table(dbx.tbl("gold_h3_geobosques")), "h3", "left") if HAS_GB else gh
wide = (base_h3.join(wide, "h3", "left")
        .withColumn("d_ndvi", F.col(f"ndvi_{Y1}") - F.col(f"ndvi_{Y0}"))
        .withColumn("d_pert", F.col(f"pert_{Y1}") - F.col(f"pert_{Y0}")))
feat = wide.toPandas().pipe(dbx.clean)
print(f"Hexágonos: {len(feat):,} | con dato en {Y1}: {feat[f'ndvi_{Y1}'].notna().sum():,}")

## 3. Covariables geográficas

Hipótesis de trabajo sobre la minería aluvial:
- se concentra **cerca de los ríos** (el oro está en los sedimentos),
- se expande **desde la Carretera Interoceánica** (acceso),
- es menor **dentro de áreas protegidas** y se presiona sobre su **zona de amortiguamiento** (SERNANP).

In [ ]:
import h3, numpy as np, geopandas as gpd
from shapely.geometry import Point

lat_lon = np.array([h3.cell_to_latlng(c) for c in feat.h3])
feat["lat"], feat["lon"] = lat_lon[:, 0], lat_lon[:, 1]
pts = gpd.GeoDataFrame(feat, geometry=gpd.points_from_xy(feat.lon, feat.lat), crs="EPSG:4326").to_crs(C.CRS_UTM)
feat["x_km"], feat["y_km"] = pts.geometry.x / 1000, pts.geometry.y / 1000

vec = Path(VOL) / "bronze" / "vector"
def load(name):
    p = vec / f"{name}.geojson"
    return gpd.read_file(p).to_crs(C.CRS_UTM) if p.exists() else None

vias, rios, dist = load("vias"), load("rios"), load("distritos_madre_de_dios")
anp = load("areas_protegidas_sernanp")          # oficial SERNANP (data/manual); si no existe, OSM
if anp is None:
    anp = load("areas_protegidas")
za = load("zonas_amortiguamiento")   # SERNANP, opcional (data/manual)
if vias is not None:
    principales = vias[vias["highway"].isin(["trunk", "primary"])] if "highway" in vias else vias
    feat["dist_via_km"] = pts.distance(principales.union_all()).values / 1000
if rios is not None:
    feat["dist_rio_km"] = pts.distance(rios.union_all()).values / 1000
if anp is not None and len(anp):
    feat["en_anp"] = pts.within(anp.union_all()).astype(int).values
    feat["dist_anp_km"] = pts.distance(anp.union_all()).values / 1000
if za is not None and len(za):
    feat["en_za"] = pts.within(za.union_all()).astype(int).values
    feat["dist_za_km"] = pts.distance(za.union_all()).values / 1000
if dist is not None:
    feat["distrito"] = gpd.sjoin(pts[["geometry"]], dist[["distrito", "geometry"]], how="left",
                                 predicate="within")["distrito"].groupby(level=0).first().values
dbx.save_table(spark.createDataFrame(feat.drop(columns=[], errors="ignore")), "gold_h3_features")
feat.describe().T.round(3)

## 4. Exportación del dataset limpio para la demostración

In [ ]:
from geocloud import spatial
out = Path(VOL) / "gold"; out.mkdir(parents=True, exist_ok=True)
feat.to_parquet(out / "h3_features.parquet", index=False)
feat.to_csv(out / "h3_features.csv", index=False, encoding="utf-8")
import shutil, tempfile
tmp = Path(tempfile.mkdtemp()) / "h3_features.geojson"      # GDAL escribe en disco local y luego se copia al Volume
spatial.h3_to_geodataframe(feat).to_file(tmp, driver="GeoJSON")
shutil.copyfile(tmp, out / "h3_features.geojson")
print(sorted(p.name for p in out.iterdir()))

## 5. AEDE: distribución y mapas temáticos

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from geocloud import spatial

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas().pipe(dbx.clean)
YEARS = sorted(int(c.split("_")[1]) for c in feat.columns if c.startswith("ndvi_") and c.split("_")[1].isdigit())
Y0, Y1 = YEARS[0], YEARS[-1]
gdf = spatial.h3_to_geodataframe(feat, crs_out=C.CRS_UTM)
print(f"Hexágonos: {len(gdf):,} | años: {YEARS}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for y in YEARS:
    ax[0].hist(feat[f"ndvi_{y}"].dropna(), bins=60, alpha=0.5, label=str(y), density=True)
ax[0].set_title("NDVI por hexágono"); ax[0].legend()
ax[1].hist(feat["d_ndvi"].dropna(), bins=80, color="#8c2d04"); ax[1].axvline(0, c="k", lw=0.8)
ax[1].set_title(f"Cambio de NDVI {Y0}-{Y1}")
REF = "frac_perdida_oficial" if "frac_perdida_oficial" in feat else "frac_perdida_hansen"
ax[2].scatter(feat[REF], feat["d_pert"], s=2, alpha=0.3)
ax[2].set_xlabel(f"{REF} (fracción del hexágono)"); ax[2].set_ylabel("Δ fracción perturbada (Sentinel-2)")
ax[2].set_title(f"Validación Sentinel-2 vs referencia, r = {feat[[REF,'d_pert']].corr().iloc[0,1]:.2f}")
plt.tight_layout(); plt.savefig(FIG / "02_distribuciones.png", dpi=150); plt.show()

In [ ]:
def mapa(ax, col, cmap, title, vmin=None, vmax=None):
    gdf.plot(column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0, legend=True,
             missing_kwds={"color": "lightgrey"}, legend_kwds={"shrink": 0.6})
    if rios is not None: rios.plot(ax=ax, color="#1f78b4", lw=0.6)
    if vias is not None: vias.plot(ax=ax, color="black", lw=0.8)
    if anp is not None: anp.boundary.plot(ax=ax, color="purple", lw=0.8)
    if za is not None: za.boundary.plot(ax=ax, color="darkorange", lw=0.8, ls="--")
    ax.set_xlim(C.GRID_BOUNDS_UTM[0], C.GRID_BOUNDS_UTM[2]); ax.set_ylim(C.GRID_BOUNDS_UTM[1], C.GRID_BOUNDS_UTM[3])
    ax.set_title(title); ax.set_axis_off()

vec = Path(VOL) / "bronze" / "vector"
rd = lambda n: gpd.read_file(vec / f"{n}.geojson").to_crs(C.CRS_UTM) if (vec / f"{n}.geojson").exists() else None
rios, vias, za = rd("rios"), rd("vias"), rd("zonas_amortiguamiento")
anp = rd("areas_protegidas_sernanp")
if anp is None:
    anp = rd("areas_protegidas")

fig, ax = plt.subplots(2, 2, figsize=(16, 11))
mapa(ax[0, 0], f"ndvi_{Y0}", "RdYlGn", f"NDVI {Y0}", 0, 0.9)
mapa(ax[0, 1], f"ndvi_{Y1}", "RdYlGn", f"NDVI {Y1}", 0, 0.9)
mapa(ax[1, 0], "d_ndvi", "RdBu", f"Cambio de NDVI {Y0}-{Y1}", -0.6, 0.6)
if "frac_perdida_oficial" in feat:
    mapa(ax[1, 1], "frac_perdida_oficial", "Reds", f"Pérdida antrópica oficial Geobosques {Y0+1}-{Y1}", 0, 1)
else:
    mapa(ax[1, 1], "frac_perdida_hansen", "Reds", f"Pérdida de bosque Hansen {Y0+1}-{Y1}", 0, 1)
plt.tight_layout(); plt.savefig(FIG / "02_mapas.png", dpi=150); plt.show()

In [ ]:
bins = [0, 0.5, 1, 2, 3, 5, 8, 12, 20, np.inf]
if "dist_rio_km" in feat:
    feat["banda_rio"] = pd.cut(feat["dist_rio_km"], bins)
    t = feat.groupby("banda_rio", observed=True).agg(hexagonos=("h3", "size"),
                                                      d_pert=("d_pert", "mean"),
                                                      perdida_hansen=("frac_perdida_hansen", "mean"),
                                                      **({"perdida_oficial": ("frac_perdida_oficial", "mean"),
                                                          "perdida_natural": ("frac_perdida_natural_2025", "mean")}
                                                         if "frac_perdida_oficial" in feat else {}))
    ax = t.drop(columns="hexagonos").plot.bar(figsize=(11, 4))
    ax.set_xlabel("Distancia al río más cercano (km)"); ax.set_ylabel("Fracción media")
    ax.set_title("Gradiente de perturbación con la distancia al río")
    plt.tight_layout(); plt.savefig(FIG / "02_gradiente_rio.png", dpi=150); plt.show()
    display_t = t.round(3); print(display_t)